# Optimal Transport (Wasserstein) -- Interactive Notebook

**Concept 41** of the math-foundations learning map.

We explore the Wasserstein-1 distance between empirical 1D distributions using
the closed-form quantile (sorted-sample) formula, and contrast its behaviour
with the Kullback-Leibler divergence on shifting Gaussians and on disjoint
uniforms.


## 1. The 1D closed form

For probability measures $\mu, \nu$ on $\mathbb{R}$ with quantile functions
$F_\mu^{-1}, F_\nu^{-1}$,
$$ W_p(\mu, \nu)^p \;=\; \int_0^1 |F_\mu^{-1}(u) - F_\nu^{-1}(u)|^p\,du. $$

For two equal-sized samples this becomes: sort both, then average $|x_{(i)} - y_{(i)}|^p$.


In [ ]:
import math
import random

def w1_empirical_1d(xs, ys):
    assert len(xs) == len(ys)
    return sum(abs(a - b) for a, b in zip(sorted(xs), sorted(ys))) / len(xs)

def gaussian_sample(mean, std, n, seed):
    rng = random.Random(seed)
    return [rng.gauss(mean, std) for _ in range(n)]


## 2. Translation of a Gaussian: $W_1(\mathcal{N}(0,1), \mathcal{N}(m,1)) \approx |m|$

The optimal coupling is $Y = X + m$, which gives $W_1 = |m|$ exactly. The
empirical estimate should track this closely.


In [ ]:
n = 8000
base = gaussian_sample(0.0, 1.0, n, seed=1)

print(f"{'m':>6} {'W_1':>10} {'true |m|':>10}")
for m in [0.0, 0.25, 0.5, 1.0, 2.0, 5.0]:
    shifted = gaussian_sample(m, 1.0, n, seed=2)
    print(f"{m:>6.2f} {w1_empirical_1d(base, shifted):>10.4f} {abs(m):>10.2f}")


## 3. KL diverges, $W_1$ stays finite (disjoint supports)

Take $\mu = \mathrm{Unif}[0, 1]$ and $\nu_s = \mathrm{Unif}[s, s+1]$. For
$s \geq 1$ the supports are disjoint; KL is $+\infty$. But $W_1(\mu, \nu_s) = s$
(rigid translation by $s$).


In [ ]:
def w1_uniforms(a_lo, a_hi, b_lo, b_hi, n=4000, seed=3):
    rng = random.Random(seed)
    xs = [rng.uniform(a_lo, a_hi) for _ in range(n)]
    ys = [rng.uniform(b_lo, b_hi) for _ in range(n)]
    return w1_empirical_1d(xs, ys)

print(f"{'shift s':>9} {'W_1 est':>10} {'KL':>10}")
for s in [0.0, 0.5, 1.0, 2.0, 5.0]:
    w = w1_uniforms(0.0, 1.0, s, s + 1.0)
    k = '+inf' if s >= 1.0 else 'finite'
    print(f"{s:>9.2f} {w:>10.4f} {k:>10}")


## 4. The optimal coupling is the sorted (monotone) one

We can verify experimentally that any other pairing produces strictly larger
total transport cost. Below we compare the sorted coupling against random
permutations.


In [ ]:
rng = random.Random(7)
xs = [rng.gauss(0, 1) for _ in range(200)]
ys = [rng.gauss(2, 1) for _ in range(200)]

sorted_cost = w1_empirical_1d(xs, ys)
ys_perm = ys[:]
random_costs = []
for _ in range(50):
    rng.shuffle(ys_perm)
    random_costs.append(sum(abs(a - b) for a, b in zip(xs, ys_perm)) / len(xs))

print(f"sorted (monotone) coupling cost: {sorted_cost:.4f}")
print(f"min over 50 random pairings    : {min(random_costs):.4f}")
print(f"mean of 50 random pairings     : {sum(random_costs)/len(random_costs):.4f}")
print("-> sorted coupling is the cheapest, as the 1D theorem predicts.")


## 5. Two point masses

$W_p(\delta_0, \delta_1) = 1$ for every $p \geq 1$. The unique coupling is
the Dirac at $(0, 1)$ in $\mathbb{R}^2$. Compare: $D_{\mathrm{KL}}(\delta_0 \,\|\, \delta_1) = +\infty$
and total variation distance $= 1$ regardless of how far apart the points are.


In [ ]:
for d in [0.1, 1.0, 10.0]:
    xs = [0.0] * 100
    ys = [d] * 100
    print(f"W_1(delta_0, delta_{d}) = {w1_empirical_1d(xs, ys):.4f}   (= {d})")


## 6. Take-aways

- $W_1$ is **geometry-aware**: shifting a distribution by $m$ moves it by exactly
  distance $|m|$ in $W_1$. KL doesn't even know there is a metric.
- $W_1$ stays finite when **supports are disjoint**; KL blows up. This is exactly
  why Wasserstein GANs (Arjovsky et al. 2017) train where JS-GANs collapse.
- In 1D the **sorted coupling** is optimal. In higher dimensions one needs a
  linear program (Sinkhorn / network simplex), but the duality
  $W_1 = \sup_{\|f\|_{\mathrm{Lip}} \leq 1} \int f\,d(\mu - \nu)$ makes the problem
  amenable to neural critics.
- Flow-matching trajectories $x_t = (1-t) x_0 + t x_1$ are exactly $W_2$
  geodesics in Euclidean space (McCann displacement interpolation). Manifold-aware
  papers like ELF replace these straight lines with curved OT geodesics.
